# Linear AR + Conformal Prediction (sanity baseline)

Trains a Linear Autoregression model per `(ticker, crisis regime)` split, wraps it with Split CP and ACI, walks forward through the test window, and writes daily interval arrays to `results/`.

Linear AR is not in the proposal's 6 evaluation methods — it's kept here as a **sanity baseline** because it fits in milliseconds, which makes it useful for:
1. Debugging pipeline changes (data load, splits, save/load) without waiting for LSTM/Transformer training.
2. A weak lower-bound reference point: any CP wrapper should at least improve over naive Linear AR + CP.

Two sections mirror the other per-model notebooks:
1. **Smoke test** — one split, both wrappers.
2. **Full grid** — every `(ticker, regime)` x `{Split CP, ACI}`. Fast enough that `RUN_FULL_GRID` defaults to `True`.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / "requirements.txt").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break


In [ ]:
import numpy as np
import pandas as pd

from src.analysis._shared import (
    ensure_data,
    iterate_experiments,
    walk_forward,
    make_wrapper,
    save_intervals,
)
from src.data.data import Ticker
from src.models.linear_ar import LinearARModel


In [ ]:
NUM_LAGS = 5
CONFIDENCE_LEVEL = 0.90  # alpha = 0.10, matches Metrics.md
ACI_STEP_SIZE = 0.01

MODEL_NAME = "linear_ar"

np.random.seed(0)


## Load data

Downloads the full `ALL_TICKERS` panel from 2003-01-01 onward (cached under `datasets/all_tickers.csv`). Re-running is a no-op once the CSV exists.


In [ ]:
data = ensure_data()
print(f"{len(data):,} rows  |  {data.index.min().date()} -> {data.index.max().date()}")
data.head()


## Smoke test: one split

Picks the 2020 x MSFT split, runs Split CP and ACI, saves intervals. Prints per-regime breakdown (pre-crisis calm / COVID window / post-crisis). Use this to verify the pipeline before the full grid.


In [ ]:
smoke_split = next(
    s for s in iterate_experiments()
    if s.regime == "2020" and s.ticker == Ticker.Microsoft
)
smoke_split


In [ ]:
train_df = smoke_split.slice_train(data)
calib_df = smoke_split.slice_calib(data)
full_df  = smoke_split.slice_full(data)

print(f"train: {len(train_df):>5} rows  |  calib: {len(calib_df):>4} rows")


In [ ]:
def summarise(intervals, label):
    widths = intervals["u"] - intervals["l"]
    covered = (intervals["r"] >= intervals["l"]) & (intervals["r"] <= intervals["u"])
    inf_count = np.isinf(widths).sum()

    pre  = intervals[intervals.index < "2020-02-20"]
    mid  = intervals[(intervals.index >= "2020-02-20") & (intervals.index <= "2020-04-30")]
    post = intervals[intervals.index > "2020-04-30"]

    def cov(df):
        if len(df) == 0: return float("nan")
        return ((df["r"] >= df["l"]) & (df["r"] <= df["u"])).mean()

    print(f"{label:5s}  cov={covered.mean():.3f}  mean_w={widths.mean():.4f}  inf_days={inf_count}")
    print(f"       pre-crisis  cov={cov(pre):.3f}  ({len(pre)} rows, mean_w={(pre['u']-pre['l']).mean():.4f})")
    print(f"       COVID       cov={cov(mid):.3f}  ({len(mid)} rows, mean_w={(mid['u']-mid['l']).mean():.4f})")
    print(f"       post-crisis cov={cov(post):.3f}  ({len(post)} rows, mean_w={(post['u']-post['l']).mean():.4f})")


In [ ]:
split_wrapper = make_wrapper(
    model_class=LinearARModel,
    label_col=smoke_split.label_col,
    num_lags=NUM_LAGS,
    confidence_level=CONFIDENCE_LEVEL,
    step_size=None,
)
split_wrapper.fit_and_calibrate(train_df, calib_df)

split_intervals = walk_forward(
    split_wrapper, full_df,
    test_start=smoke_split.test_start,
    test_end=smoke_split.test_end,
    is_adaptive=False,
)
save_intervals(split_intervals, MODEL_NAME, "split", smoke_split.ticker, smoke_split.regime)
summarise(split_intervals, "SPLIT")


In [ ]:
aci_wrapper = make_wrapper(
    model_class=LinearARModel,
    label_col=smoke_split.label_col,
    num_lags=NUM_LAGS,
    confidence_level=CONFIDENCE_LEVEL,
    step_size=ACI_STEP_SIZE,
)
aci_wrapper.fit_and_calibrate(train_df, calib_df)

aci_intervals = walk_forward(
    aci_wrapper, full_df,
    test_start=smoke_split.test_start,
    test_end=smoke_split.test_end,
    is_adaptive=True,
)
save_intervals(aci_intervals, MODEL_NAME, "aci", smoke_split.ticker, smoke_split.regime)
summarise(aci_intervals, "ACI")


## Full grid

Every `(ticker, regime)` split x `{Split CP, ACI}`. LinearAR fits in milliseconds so this runs in well under a minute — default on.


In [ ]:
RUN_FULL_GRID = True
MIN_TRAIN_ROWS = NUM_LAGS + 50  # skip splits with too little history (e.g., Zoom in 2008)

if RUN_FULL_GRID:
    results = []
    for split in iterate_experiments():
        train_df = split.slice_train(data)
        calib_df = split.slice_calib(data)
        full_df  = split.slice_full(data)

        if len(train_df) < MIN_TRAIN_ROWS:
            print(f"skip {split.regime} {split.ticker.value:5s}: only {len(train_df)} train rows")
            continue

        for wrapper_name, step_size in [("split", None), ("aci", ACI_STEP_SIZE)]:
            wrapper = make_wrapper(
                model_class=LinearARModel,
                label_col=split.label_col,
                num_lags=NUM_LAGS,
                confidence_level=CONFIDENCE_LEVEL,
                step_size=step_size,
            )
            wrapper.fit_and_calibrate(train_df, calib_df)
            intervals = walk_forward(
                wrapper, full_df,
                test_start=split.test_start,
                test_end=split.test_end,
                is_adaptive=(step_size is not None),
            )
            save_intervals(intervals, MODEL_NAME, wrapper_name, split.ticker, split.regime)

            covered = (intervals["r"] >= intervals["l"]) & (intervals["r"] <= intervals["u"])
            widths = intervals["u"] - intervals["l"]
            results.append({
                "regime": split.regime,
                "ticker": split.ticker.value,
                "wrapper": wrapper_name,
                "coverage": covered.mean(),
                "mean_width": widths.mean(),
                "inf_days": int(np.isinf(widths).sum()),
            })
            print(f"  {split.regime} {split.ticker.value:5s} {wrapper_name:5s}: cov={covered.mean():.3f}  mean_w={widths.mean():.4f}")

    pd.DataFrame(results)
